# Human 배치 — 학습 게임 수별 학습곡선
## 1. 왜 / 무엇
데이터 부족이 병목인지 확인하기 위해 기존 Train149게임을25/50/75/100%로 늘려 비교합니다.
게임 단위로38/75/112/149게임을 중첩 선택하고, 모두 같은1,530 optimizer 업데이트를 사용합니다.
작은 조건은 Smoke가 아니라 정식 실험 조건입니다. 신규 로그를 수집하지 않습니다.

**이번 그래프의 가로축은 epoch가 아니라 학습 게임 수입니다.** 각 모델이 같은 예산을 사용한 뒤 Train/Validation NLL과 정확도를 계산합니다.
[실험 명세](../../../ai-harness/ml-experiments/experiments/human-placement-game-count-learning-curve.md)


## 2. coverage와 예측력을 구분하기
작은 Train에서 배치 A/B만 보았고 큰 Train에서 A/B/C를 봤다면, C 정답을 맞힐 수 있게 된 것만으로 점수가 오를 수 있습니다.

1. 전체 Validation: 처음 보는 정답도 실패로 포함합니다. 실제 pipeline 성능입니다.
2. 공통 정답 구간: 모든 모델이 학습한 정답 A/B인 **같은 구간**만 비교합니다. 후보 class 수는 여전히 모델마다 다릅니다.
3. 공통 어휘 projection: 위 구간에서 확률 열도 A/B로 제한해 같은 정답/경쟁 공간을 비교합니다. 낙관적인 진단이며 전체 성능을 대신하지 않습니다.

모든 scaler/어휘는 각 subset의 Train만 사용합니다. 어휘가 늘면 출력층 크기도 늘므로 순수한 고정 출력층 데이터량 실험은 아닙니다.


In [ ]:
import contextlib
import hashlib
import io
import json
import math
import os
import platform
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-game-count-learning-curve"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_game_count_learning_curve.ipynb"
PREVIOUS_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_l2_regularization.ipynb"
PREVIOUS_DEFS_SHA = "9112dedc65f7b4d5f04ef2f0107561914be53da022f7f7e51dc7e50d9ce60cbd"
REFERENCE_PATH = ROOT / "data/ml-experiments/human-placement-l2-regularization/full-report-run1.json"
REFERENCE_SHA = "1b3d28aca7319265e8a73c06629ca2473d6dd7597b2967ccf78be5135491779e"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(PREVIOUS_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == PREVIOUS_DEFS_SHA
previous = {"__name__": "learning_curve_previous"}
for c in selected:
    exec(compile(c["source"], str(PREVIOUS_NOTEBOOK), "exec"), previous)
shared = previous["shared"]
shared["EXPERIMENT_ID"], shared["SOURCE_NOTEBOOK"] = EXPERIMENT_ID, SOURCE_NOTEBOOK
shared["base"]["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "threadpool_limits",
             "report_path", "publish_report", "close_numeric", "make_model",
             "rank_probabilities", "evaluate_rankings", "predict_probabilities"):
    globals()[name] = previous[name]
FRACTIONS = (.25, .5, .75, 1.0)
BATCH_SIZE, FULL_PASSES, ALPHA = 128, 30, .0001
GAME_ORDER_SEED, PARTIAL_SEED = "human-placement-games-v1", 1729


In [ ]:
def make_subsets(train):
    require(train and all(r["split"] == "train" for r in train), "SUBSET_TRAIN_ONLY")
    games = sorted({r["gameId"] for r in train}, key=lambda g: (
        hashlib.sha256((GAME_ORDER_SEED + "|" + g).encode()).hexdigest(), g))
    subsets = []
    for fraction in FRACTIONS:
        selected_games = set(games[:math.ceil(len(games) * fraction)])
        rows = [r for r in train if r["gameId"] in selected_games]
        subsets.append({"fraction": fraction, "gameIds": sorted(selected_games), "rows": rows})
    require(all(set(a["gameIds"]) <= set(b["gameIds"]) for a, b in zip(subsets, subsets[1:])),
            "SUBSETS_NOT_NESTED")
    return subsets

def update_plan(n_rows, updates):
    require(type(n_rows) is int and n_rows >= BATCH_SIZE and type(updates) is int and updates > 0,
            "INVALID_UPDATE_PLAN")
    batches = math.ceil(n_rows / BATCH_SIZE)
    passes, remaining = divmod(updates, batches)
    partial_rows = remaining * BATCH_SIZE
    return {"optimizerUpdates": updates, "batchesPerFullPass": batches, "fullPasses": passes,
            "partialPassRows": partial_rows, "partialFitCalls": passes + int(bool(partial_rows)),
            "rowPresentations": passes * n_rows + partial_rows,
            "equivalentPasses": (passes * n_rows + partial_rows) / n_rows}

def fit_updates(train_x, labels, n_classes, updates):
    """Fit Train only, using full passes plus a deterministic partial pass."""
    plan = update_plan(len(labels), updates)
    require(len(train_x) == len(labels), "TRAIN_LENGTH_MISMATCH")
    model = make_model(ALPHA)
    losses, actual_updates, presentations = [], 0, 0
    for call in range(plan["partialFitCalls"]):
        if call < plan["fullPasses"]:
            x, y = train_x, labels
        else:
            indices = np.random.default_rng(PARTIAL_SEED).permutation(len(labels))[:plan["partialPassRows"]]
            x, y = train_x[indices], labels[indices]
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.partial_fit(x, y, classes=np.arange(n_classes) if call == 0 else None)
        require(not caught, "UNEXPECTED_FIT_WARNING: " + "; ".join(
            f"{w.category.__name__}: {w.message}" for w in caught))
        require(np.isfinite(model.loss_), "NONFINITE_TRAIN_LOSS")
        losses.append(float(model.loss_))
        actual_updates += math.ceil(len(y) / BATCH_SIZE)
        presentations += len(y)
        if (call + 1) % max(1, plan["partialFitCalls"] // 5) == 0 or call + 1 == plan["partialFitCalls"]:
            print(f"updates={actual_updates}/{updates}; row presentations={presentations}")
    require(actual_updates == updates and presentations == plan["rowPresentations"]
            and int(model.t_) == presentations, "UPDATE_BUDGET_MISMATCH")
    require(all(np.isfinite(a).all() for a in (*model.coefs_, *model.intercepts_)), "NONFINITE_MODEL")
    return model, losses, {**plan, "observedRowPresentations": int(model.t_)}


In [ ]:
def project_common(rows, probability, vocabulary, common):
    """Evaluate common targets against the same Train-defined class space."""
    ids = {key: i for i, key in enumerate(vocabulary)}
    require(common and set(common) <= set(ids), "COMMON_VOCABULARY_INVALID")
    common_set = set(common)
    row_ids = [i for i, r in enumerate(rows) if r["targetKey"] in common_set]
    require(row_ids, "EMPTY_COMMON_SCOPE")
    columns = [ids[key] for key in common]
    return evaluate_rankings(rank_probabilities([rows[i] for i in row_ids],
        probability[np.ix_(row_ids, columns)], common))

def train_point(train, validation, common, updates):
    vocabulary = shared["train_vocabulary"](train)
    ids = {key: i for i, key in enumerate(vocabulary)}
    raw = np.stack([r["x"] for r in train])
    scaler = shared["base"]["StandardScaler"]().fit(raw)
    x = shared["masked_features"](scaler.transform(raw), "board_piece")
    vx = shared["masked_features"](scaler.transform(np.stack([r["x"] for r in validation])), "board_piece")
    labels = np.array([ids[r["targetKey"]] for r in train], dtype=int)
    model, losses, budget = fit_updates(x, labels, len(vocabulary), updates)
    tp = predict_probabilities(model, x, len(vocabulary))
    vp = predict_probabilities(model, vx, len(vocabulary))
    train_ranked = rank_probabilities(train, tp, vocabulary)
    validation_ranked = rank_probabilities(validation, vp, vocabulary)
    common_set = set(common)
    common_rows = [r for r, source in zip(validation_ranked, validation) if source["targetKey"] in common_set]
    return {"trainAll": evaluate_rankings(train_ranked), "validationAll": evaluate_rankings(validation_ranked),
        "validationCommonFullVocabulary": evaluate_rankings(common_rows),
        "trainCommonProjection": project_common(train, tp, vocabulary, common),
        "validationCommonProjection": project_common(validation, vp, vocabulary, common),
        "trainingLossPerPartialFit": losses, "budget": budget,
        "coefficientL2": float(np.sqrt(sum(np.square(a).sum() for a in model.coefs_))),
        "classCount": len(vocabulary), "parameterCount": sum(a.size for a in (*model.coefs_, *model.intercepts_)),
        "vocabularySha256": hashlib.sha256(json_bytes(vocabulary)).hexdigest(),
        "scaler": {"fitSegments": int(scaler.n_samples_seen_), "mean": scaler.mean_.tolist(),
                   "scale": scaler.scale_.tolist()}}

def load_reference():
    require(sha256_file(REFERENCE_PATH) == REFERENCE_SHA, "REFERENCE_SHA_MISMATCH")
    ref = json.loads(REFERENCE_PATH.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    require(ref["experimentId"] == "human-placement-l2-regularization" and ref["runMode"] == "full"
            and ref["fullCohortCovered"] is True and ref["baselineReproduced"] is True
            and ref["sourceDefinitionSha256"] == PREVIOUS_DEFS_SHA, "REFERENCE_CONTRACT_MISMATCH")
    return ref

def check_full_baseline(point, reference):
    expected = reference["results"]["alpha_0.0001"]
    require(close_numeric(point["validationAll"], expected["result"])
            and close_numeric(point["trainAll"]["segmentMicro"], expected["history"][-1]["train"])
            and close_numeric(point["trainingLossPerPartialFit"], expected["trainingLoss"])
            and close_numeric(point["coefficientL2"], expected["coefficientL2"])
            and close_numeric(point["scaler"]["mean"], reference["training"]["scalerMean"])
            and close_numeric(point["scaler"]["scale"], reference["training"]["scalerScale"])
            and point["vocabularySha256"] == reference["vocabulary"]["sha256"], "BASELINE_REPRODUCTION_FAILED")


In [ ]:
def run_full():
    require(FRACTIONS == (.25, .5, .75, 1.0) and BATCH_SIZE == 128 and FULL_PASSES == 30
            and ALPHA == .0001 and GAME_ORDER_SEED == "human-placement-games-v1" and PARTIAL_SEED == 1729,
            "EXPERIMENT_SETTINGS_CHANGED")
    require((platform.python_version(), np.__version__, shared["base"]["sklearn"].__version__)
            == ("3.12.13", "2.5.1", "1.9.0"), "PINNED_ENVIRONMENT_REQUIRED")
    source_sha = definition_sha(SOURCE_NOTEBOOK)
    reference = load_reference()
    data, watched = shared["load_snapshots"]()
    subsets = make_subsets(data["train"])
    require([len(s["gameIds"]) for s in subsets] == [38, 75, 112, 149], "SUBSET_GAME_COUNTS_CHANGED")
    catalogues = [set(shared["train_vocabulary"](s["rows"])) for s in subsets]
    common = sorted(set.intersection(*catalogues))
    require(len(common) >= 2, "COMMON_VOCABULARY_TOO_SMALL")
    common_set = set(common)
    common_validation = [r for r in data["validation"] if r["targetKey"] in common_set]
    require(common_validation, "EMPTY_COMMON_SCOPE")
    common_ids = sorted(r["segmentId"] for r in common_validation)
    updates = math.ceil(len(data["train"]) / BATCH_SIZE) * FULL_PASSES
    require(updates == 1530, "FULL_UPDATE_BUDGET_CHANGED")
    results = {}
    with threadpool_limits(limits=1):
        for subset in [subsets[-1], *subsets[:-1]]:
            fraction, rows = subset["fraction"], subset["rows"]
            print(f"fraction={fraction:g}, games={len(subset['gameIds'])}, segments={len(rows)}, updates={updates}")
            point = train_point(rows, data["validation"], common, updates)
            require(point["budget"]["optimizerUpdates"] == updates
                    and point["validationAll"]["segmentMicro"]["segments"] == 1516
                    and point["validationCommonProjection"]["segmentMicro"]["segments"] == len(common_ids)
                    and point["validationCommonFullVocabulary"]["segmentMicro"]["segments"] == len(common_ids),
                    "EVALUATION_DENOMINATOR_CHANGED")
            if fraction == 1.0:
                check_full_baseline(point, reference)
                print("100% baseline reproduced within 1e-12. Continuing smaller Train conditions.")
            results[f"fraction_{int(fraction * 100)}"] = {
                "fraction": fraction, "gameIds": subset["gameIds"], "games": len(subset["gameIds"]),
                "segments": len(rows), **point}
    require(all(sha256_file(p) == h for p, h in watched.items())
            and sha256_file(REFERENCE_PATH) == REFERENCE_SHA
            and definition_sha(SOURCE_NOTEBOOK) == source_sha
            and definition_sha(PREVIOUS_NOTEBOOK) == PREVIOUS_DEFS_SHA
            and definition_sha(previous["SNAPSHOT_NOTEBOOK"]) == previous["SNAPSHOT_DEFS_SHA"]
            and definition_sha(shared["BASE_NOTEBOOK"]) == shared["BASE_DEFS_SHA"], "INPUT_OR_CODE_CHANGED")
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": "full",
        "executionPolicy": "full-only", "fullCohortCovered": True, "policyDecisionEligible": True,
        "canonicalResultEligible": False, "productionIntegration": False, "inferenceEligible": False,
        "wholePolicyEvidence": False, "legalityChecked": False, "reachableCandidatesGenerated": False,
        "snapshotObservation": "unverified", "analysisStatus": "learning_curve_requires_review",
        "modelFitCount": 4, "smokeFitCount": 0, "fractions": list(FRACTIONS), "results": results,
        "optimizerUpdateBudget": updates, "gameOrderSeed": GAME_ORDER_SEED, "partialPassSeed": PARTIAL_SEED,
        "commonScope": {"classCount": len(common), "vocabularySha256": hashlib.sha256(json_bytes(common)).hexdigest(),
            "validationSegments": len(common_ids), "validationGames": len({r["gameId"] for r in common_validation}),
            "validationSegmentIdsSha256": hashlib.sha256(json_bytes(common_ids)).hexdigest(),
            "definitionUsesTrainOnly": True},
        "baselineReproduced": True, "baselineTolerance": 1e-12, "subsetSelectionUsesValidation": False,
        "validationUsedForFit": False, "validationUsedForModelSelection": False, "fitWarnings": 0,
        "scalerAndVocabularyFitScope": "each_train_subset_only",
        "validationRole": "reused_development_comparison_not_independent_test",
        "provenance": reference["provenance"], "epochMetricContract": reference["epochMetricContract"],
        "environment": reference["environment"], "training": {"alpha": ALPHA, "batchSize": BATCH_SIZE,
            "hiddenSizes": [64], "seed": 1729, "learningRate": .001, "optimizer": "adam",
            "fixedOutputHead": False, "convergenceAssessed": False},
        "sourceDefinitionSha256": source_sha, "previousDefinitionsSha256": PREVIOUS_DEFS_SHA,
        "referenceSha256": REFERENCE_SHA, "sourceUnchanged": True, "testFileRead": False, "rawFileRead": False,
        "newConditionRepeatDeterminismChecked": False}


In [ ]:
def internal_checks():
    with contextlib.redirect_stdout(io.StringIO()):
        previous["internal_checks"]()
    rows = [{"split": "train", "gameId": f"G{i}", "targetKey": "square|a"} for i in range(8)]
    rows.append({"split": "train", "gameId": "G0", "targetKey": "square|b"})
    a, b = make_subsets(rows), make_subsets(list(reversed(rows)))
    assert [s["gameIds"] for s in a] == [s["gameIds"] for s in b]
    assert [len(s["gameIds"]) for s in a] == [2, 4, 6, 8]
    assert all(set(x["gameIds"]) <= set(y["gameIds"]) for x, y in zip(a, a[1:]))
    for s in a:
        assert {r["gameId"] for r in s["rows"]} == set(s["gameIds"])
        assert len([r for r in s["rows"] if r["gameId"] == "G0"]) in (0, 2)
    assert update_plan(6417, 1530)["fullPasses"] == 30 and update_plan(6417, 1530)["partialPassRows"] == 0
    assert update_plan(257, 5)["partialPassRows"] == 256 and update_plan(257, 5)["rowPresentations"] == 513
    for n_rows, updates in ((127, 1), (257, 0), (True, 1), (257, True)):
        try:
            update_plan(n_rows, updates)
        except ValueError:
            pass
        else:
            raise AssertionError("INVALID_BUDGET_ACCEPTED")
    vocab, common = ["square|a", "square|b", "square|c"], ["square|a", "square|b"]
    evaluation_rows = [{"gameId": "Val", "pieceType": "square", "targetKey": k} for k in ("square|a", "square|unseen")]
    probability = np.array([[.3, .2, .5], [.3, .2, .5]])
    all_result = evaluate_rankings(rank_probabilities(evaluation_rows, probability, vocab))
    projected = project_common(evaluation_rows, probability, vocab, common)
    assert all_result["segmentMicro"]["segments"] == 2 and all_result["segmentMicro"]["unknownTargetSegments"] == 1
    assert all_result["segmentMicro"]["top1"] == 0 and projected["segmentMicro"]["top1"] == 1
    assert projected["segmentMicro"]["segments"] == 1
    assert np.isclose(projected["segmentMicro"]["knownTargetNll"], -np.log(.6))
    train_x = np.arange(257 * 207, dtype=float).reshape(257, 207)
    labels = np.arange(257) % 2
    made, failure = [], None
    class FakeModel:
        coefs_, intercepts_ = [np.ones((1, 1))], [np.zeros(1)]
        def __init__(self, alpha):
            assert alpha == .0001
            self.calls, self.t_, self.updates = [], 0, 0
            made.append(self)
        def partial_fit(self, x, y, classes=None):
            indices = (x[:, 0] / 207).astype(int)
            assert len(set(indices)) == len(indices) and all(0 <= i < 257 for i in indices)
            assert np.array_equal(y, labels[indices])
            assert (classes is None) == bool(self.calls)
            if classes is not None:
                assert np.array_equal(classes, np.arange(2))
            self.calls.append(indices.tolist()); self.t_ += len(y)
            self.updates += math.ceil(len(y) / BATCH_SIZE)
            self.loss_ = np.float64(1 / len(self.calls))
            if failure == "nonfinite":
                self.loss_ = np.float64(np.nan)
            if failure == "warning":
                warnings.warn("internal-check", RuntimeWarning)
            return self
    original = globals()["make_model"]
    try:
        globals()["make_model"] = FakeModel
        with contextlib.redirect_stdout(io.StringIO()):
            model, losses, budget = fit_updates(train_x, labels, 2, 5)
            other, _, _ = fit_updates(train_x, labels, 2, 5)
            small, _, small_budget = fit_updates(train_x, labels, 2, 1)
        assert model.updates == 5 and model.t_ == 513 and losses == [1., .5]
        assert model.calls == other.calls and [len(x) for x in model.calls] == [257, 256]
        assert small.updates == 1 and small.t_ == 128 and small_budget["fullPasses"] == 0
        for failure, reason in (("nonfinite", "NONFINITE_TRAIN_LOSS"),
                                ("warning", "UNEXPECTED_FIT_WARNING: RuntimeWarning: internal-check")):
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    fit_updates(train_x, labels, 2, 5)
            except ValueError as error:
                assert str(error) == reason
            else:
                raise AssertionError("INVALID_TRAIN_STEP_ACCEPTED")
    finally:
        globals()["make_model"] = original
    return "PASS: shared contracts; nested game subsets; fixed update/partial pass budget; Train-only updates; common projection/OOV; NumPy loss and warnings. No real fit."


In [ ]:
def show_report(report):
    require(report["experimentId"] == EXPERIMENT_ID and report["baselineReproduced"], "REPORT_CONTRACT_INVALID")
    points = [report["results"][f"fraction_{int(f * 100)}"] for f in FRACTIONS]
    summary, budgets = [], []
    for p in points:
        full, projected = p["validationAll"], p["validationCommonProjection"]
        summary.append({"games": p["games"], "segments": p["segments"], "classes": p["classCount"],
            "trainMacroTop1": p["trainAll"]["gameMacro"]["top1"], "valMacroTop1": full["gameMacro"]["top1"],
            "valMacroMrr": full["gameMacro"]["mrr"], "coverage": full["segmentMicro"]["knownTargetCoverage"],
            "oov": full["segmentMicro"]["unknownTargetSegments"],
            "commonFullTop1": p["validationCommonFullVocabulary"]["gameMacro"]["top1"],
            "commonProjectedTop1": projected["gameMacro"]["top1"],
            "trainCommonNll": p["trainCommonProjection"]["segmentMicro"]["knownTargetNll"],
            "valCommonNll": projected["segmentMicro"]["knownTargetNll"]})
        budgets.append({"games": p["games"], **p["budget"], "parameterCount": p["parameterCount"]})
    table = pd.DataFrame(summary).set_index("games")
    print(report["analysisStatus"], "| one nested ordering; fixed updates; expanding Train vocabulary")
    display(table)
    display(pd.DataFrame(budgets).set_index("games"))
    display(pd.DataFrame([report["commonScope"]]))
    counts = pd.DataFrame([{"games": p["games"], "trainAllN": p["trainAll"]["segmentMicro"]["segments"],
        "trainCommonN": p["trainCommonProjection"]["segmentMicro"]["segments"],
        "valAllN": p["validationAll"]["segmentMicro"]["segments"],
        "valKnownN": p["validationAll"]["segmentMicro"]["knownNllDenominator"],
        "valCommonN": p["validationCommonProjection"]["segmentMicro"]["segments"]} for p in points])
    display(counts.set_index("games"))
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    table[["trainMacroTop1", "valMacroTop1"]].plot(marker="o", ax=axes[0, 0])
    axes[0, 0].set(title="All targets; vocabulary expands", ylabel="Game-macro Top-1", ylim=(0, 1))
    table[["commonFullTop1", "commonProjectedTop1"]].plot(marker="o", ax=axes[0, 1])
    axes[0, 1].set(title="Same common-target Validation cases", ylabel="Game-macro Top-1", ylim=(0, 1))
    table[["coverage"]].plot(marker="o", ax=axes[1, 0], legend=False)
    axes[1, 0].set(title="All Validation target coverage", ylabel="Supported fraction", ylim=(0, 1))
    table[["trainCommonNll", "valCommonNll"]].plot(marker="o", ax=axes[1, 1])
    axes[1, 1].set(title="Fixed common vocabulary projection", ylabel="Common-target NLL, no L2")
    for ax in axes.flat:
        ax.set_xlabel("Train games")
    fig.suptitle("1530 optimizer updates per condition; common projection is diagnostic, NOT full performance")
    fig.tight_layout(); plt.show(); plt.close(fig)


## 3. 내부 검사 / Full 실행
기본 모두 실행은 내부 assert만 수행합니다. 네 비율은 정식 Full 비교 조건이며 별도 Smoke 학습은 없습니다.
RUN_FULL=True와 새 FULL_REPORT 경로를 지정하고 위에서부터 실행하세요.
먼저100%로 기존30pass 결과를 재현한 뒤 작은 subset을 학습합니다. 총4모델이며 각1530업데이트입니다.


In [ ]:
RUN_FULL = False
FULL_REPORT = os.environ.get("HUMAN_PLACEMENT_GAME_COUNT_REPORT",
    "data/ml-experiments/human-placement-game-count-learning-curve/full-report-run1.json")


In [ ]:
print(internal_checks())
full_report = None
if RUN_FULL:
    destination = report_path(FULL_REPORT)
    full_report = run_full()
    publish_report(full_report, destination)
    print("Saved:", str(destination), "| SHA-256:", sha256_file(destination))
else:
    print("Full disabled. No real dataset/reference report access or neural training.")


## 4. 결과 읽기
저장 결과만 읽을 때 RUN_FULL=False를 유지하고 REPORT_TO_READ에 JSON 경로를 넣습니다.
- 전체 성능과 coverage가 같이 오르면 새 정답을 알게 된 영향이 섞여 있습니다.
- 같은 common projection 성능도 오르면 공통 문제의 예측력 개선 신호입니다.
- 네 점이 정체됐다는 이유만으로 데이터가 충분하다고 확정하거나, 증가 추세만으로 필요한 추가 게임 수를 외삽하지 않습니다.
표의 NLL은 공통 후보 공간/known 분모를 구분합니다. projection 성능은 낙관적 진단입니다.


In [ ]:
REPORT_TO_READ = None
if REPORT_TO_READ:
    path = Path(REPORT_TO_READ)
    if not path.is_absolute():
        path = ROOT / path
    loaded = json.loads(path.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    require(loaded["reportSchemaVersion"] == 1 and loaded["experimentId"] == EXPERIMENT_ID
            and loaded["runMode"] == "full" and loaded["fractions"] == list(FRACTIONS)
            and loaded["optimizerUpdateBudget"] == 1530
            and loaded["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK)
            and loaded["previousDefinitionsSha256"] == PREVIOUS_DEFS_SHA
            and loaded["referenceSha256"] == REFERENCE_SHA, "STALE_OR_DIFFERENT_REPORT")
    full_report = loaded


In [ ]:
if full_report is not None:
    show_report(full_report)


## 5. 한계 / 다음 gate
- 데이터 수를 늘리면 fitted scaler/어휘/출력 class 수도 달라집니다. 공통 평가로 일부 영향을 분리하지만 고정 출력층에서 순수한 데이터량 효과를 입증하는 실험은 아닙니다.
- 업데이트 수는 같아도 행 노출량/epoch 수/계산 시간은 같지 않습니다. 각 조건의 budget 표를 확인하세요.
- 하나의 중첩 게임 순서와 model seed이며 부분집합 구성의 우연성이 남습니다. 같은 Validation은 이미 관측한 개발 자료입니다.
- 범위 밖 Test/새 로그/원인 불명 필드는 학습에 넣지 않습니다. Legacy snapshot·합법성/도달 가능성·현재40행 호환성도 미검증입니다.
- 출력은 report뿐입니다. 다음은 사용자 Full 해석이며 추가 수집·재학습·모델 승격·runtime·commit을 자동 시작하지 않습니다.
